In [ ]:
import joblib
import pandas as pd
import firebase_admin
from firebase_admin import credentials, db
from disease_prediction import predict_disease,low_activity,very_low_activity,localized_temp_rise,moderate_heart_rate,determine_behavior

In [ ]:
# Initialize Firebase app only if it hasn't been initialized yet
try:
    firebase_admin.get_app()
except ValueError:
    cred = credentials.Certificate("backend/FireBase_Folder/credentials.json")
    firebase_admin.initialize_app(cred, {
        "databaseURL": ""
    })


In [ ]:
# Load the scaler and model for predictions later if needed.
scaler = joblib.load("backend/ML/Plk/scaler.pkl")
model = joblib.load("backend/ML/Plk/xgboost_model.pkl")

c:\cattle-health-monitoring\myenv\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [34]:
def fetch_data_from_firebase():
    """Fetch data from Firebase and return a list of test cases."""
    ref = db.reference('cattle_health_monitoring')
    data = ref.get()
    
    # Check if data is not None
    if not data:
        print("No data found in Firebase database.")
        return []
    
    # Extract only the required fields
    test_cases = []
    for key, value in data.items():
        test_case = {
            'Timestamp': value.get('Timestamp'),
            'HeartRate_BPM': value.get('HeartRate_BPM'),  
            'SkinTemp_Celsius': value.get('SkinTemp_Celsius'), 
            'Activity_X': value.get('Activity_X'),  
            'Activity_Y': value.get('Activity_Y'),  
            'Activity_Z': value.get('Activity_Z')  
        }
        test_cases.append(test_case)
    
    return test_cases

In [35]:
def push_data_to_firebase(test_case, health_status, diseases, behavior, low_activity, very_low_activity, localized_temp_rise, moderate_heart_rate):
   ref = db.reference('cattle_health_predictions')
   
   try:
       ref.push({
           'Timestamp': test_case['Timestamp'],
           'Health_Data': {
               'HeartRate_BPM': test_case['HeartRate_BPM'],
               'SkinTemp_Celsius': test_case['SkinTemp_Celsius'],
               'Activity': {
                   'Activity_X': test_case['Activity_X'],
                   'Activity_Y': test_case['Activity_Y'],
                   'Activity_Z': test_case['Activity_Z'],
               },
               'Health_Status': health_status,
               'Predicted_Diseases': diseases,
               'Behavior': behavior,
               'Activity_Analysis': {
                   'Low_Activity': low_activity,
                   'Very_Low_Activity': very_low_activity,
                   'Localized_Temp_Rise': localized_temp_rise,
                   'Moderate_Heart_Rate': moderate_heart_rate,
               }
           }
       })
       print(f"Data successfully pushed for Timestamp: {test_case['Timestamp']}")
       
   except Exception as e:
       print(f"An error occurred while pushing data to Firebase: {e}")

In [36]:
def evaluate_test_cases(test_cases):
    """Evaluate each test case and predict health status and diseases."""
    
    if not test_cases:
        print("No test cases to evaluate.")
        return
    
    feature_order = ['HeartRate_BPM', 'Activity_X', 'Activity_Y', 'Activity_Z', 'SkinTemp_Celsius']
    
    for idx, test_features in enumerate(test_cases):
        try:
            # Create DataFrame with proper order
            features_df = pd.DataFrame([test_features], columns=feature_order)
            
            # Handle missing values
            if features_df.isnull().any().any():
                print(f"Test Case {idx + 1}: Missing values detected, skipping this test case.")
                continue
            
            # Scale features
            features_scaled = scaler.transform(features_df)
            
            # Predict disease
            diseases = predict_disease(test_features)
            
            # Predict health status
            health_prediction = model.predict(features_scaled)
            
            # Display results
            health_status = 'Healthy' if health_prediction[0] == 1 else 'Unhealthy'
            print(f"Test Case {idx + 1}: {health_status} - Predicted Diseases: {', '.join(diseases)}")
            
            # Check for existing entries in Firebase based on timestamp
            existing_data_ref = db.reference('cattle_health_predictions')
            existing_data = existing_data_ref.order_by_child('Timestamp').equal_to(test_features['Timestamp']).get()
            
            if existing_data:
                print(f"Data for Timestamp {test_features['Timestamp']} already evaluated, skipping push.")
                continue
            
            # Push results to Firebase
            push_data_to_firebase(
                test_features,
                health_status,
                diseases,
                determine_behavior(test_features),
                low_activity(test_features),
                very_low_activity(test_features),
                localized_temp_rise(test_features),
                moderate_heart_rate(test_features)
            )
        
        except Exception as e:
            print(f"An error occurred while evaluating Test Case {idx + 1}: {e}")

In [37]:
test_cases = fetch_data_from_firebase()   

In [39]:
evaluate_test_cases(test_cases)  

Test Case 1: Healthy - Predicted Diseases: No specific disease detected
Data for Timestamp 2024-08-15 06:00:00 already evaluated, skipping push.
Test Case 2: Healthy - Predicted Diseases: No specific disease detected
Data for Timestamp 2024-08-15 06:05:00 already evaluated, skipping push.
Test Case 3: Healthy - Predicted Diseases: No specific disease detected
Data for Timestamp 2024-08-15 06:10:00 already evaluated, skipping push.
Test Case 4: Unhealthy - Predicted Diseases: No specific disease detected
Data for Timestamp 2024-08-15 06:15:00 already evaluated, skipping push.
Test Case 5: Healthy - Predicted Diseases: No specific disease detected
Data for Timestamp 2024-08-15 06:20:00 already evaluated, skipping push.
Test Case 6: Unhealthy - Predicted Diseases: Lameness
Data for Timestamp 2024-08-15 06:25:00 already evaluated, skipping push.
Test Case 7: Unhealthy - Predicted Diseases: No specific disease detected
Data for Timestamp 2024-08-15 06:30:00 already evaluated, skipping push.